# Phase 7 — Evaluation, Monitoring, and Reports

**Roadmap reference:** `07_Evaluation_and_Observability.ipynb`

**What this notebook does:** aggregates metrics from artifacts ALREADY
produced and verified by Phases 2–6 (parse success, human-reviewed entity
accuracy, retrieval recall, citation validity, partial-run frequency,
schema/API failures, cost/latency), generates versioned per-document
reports from Phase 5's Assessments, and scores real findings in Langfuse
by their independently-computed verification outcome — not the model's
own confidence.

**Two real bugs were found and fixed while building this phase** (not
assumed working): `parse_success_rate` was reading a stale Phase 0
manifest field that's never updated after parsing (silently reporting 0%
instead of the real 100%), and Phase 2's money/date context classification
used a whole clause instead of a window local to the specific match,
mislabeling a GST line item as `invoice_amount` just because the word
"Invoice" appeared on an adjacent line. Both are now covered by regression
tests in `tests/`.

**Known operational blocker:** OpenAI API credits were exhausted while
building this phase (`insufficient_quota`), so `schema_failure_rate`,
`api_call_failure_rate`, and `cost_and_latency_by_stage` have no fresh
calls to measure yet — they report `null` with an honest note rather than
a fabricated number. These will populate the next time any specialist or
embedding call actually runs.

**Completion gate:** a poor result can be traced to a concrete stage;
every metric has a clear definition; every report is evidence-linked and
versioned.

## 1. Load configuration

In [7]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.evaluation import build_evaluation_report
from src.contractiq.reporting import generate_document_report, generate_all_reports

manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")

Loaded manifest: 36 documents


## 2. Build the evaluation report

Every metric below is computed from a real, already-saved artifact — no
number here is estimated or invented. Where a metric genuinely cannot be
measured yet (missed-risk rate needs a hand-reviewed checklist that
doesn't exist; cost/latency needs fresh LLM calls that are currently
blocked on API credits), it reports `null` with an explicit note instead
of a fabricated value.

In [9]:
report = build_evaluation_report(manifest)

print(f"Parse success:        {report['parse_success']['rate']:.1%} "
      f"({report['parse_success']['parsed']}/{report['parse_success']['total']})")
print(f"Entity accuracy:       {report['entity_accuracy']['rate']:.1%} "
      f"({report['entity_accuracy']['correct']}/{report['entity_accuracy']['reviewed']} human-reviewed)")
print(f"Retrieval relevance:   {report['retrieval_relevance']['rate']:.1%} "
      f"(recall@{report['retrieval_relevance']['top_k']}, {report['retrieval_relevance']['evaluated']} questions)")
cv = report["citation_validity_and_finding_support"]
print(f"Citation validity:     {cv['citation_validity_rate']:.1%} "
      f"({cv['verified']}/{cv['total_findings']} findings verified, {cv['failed_verification']} failed, "
      f"across {cv['documents']} documents)")
prf = report["partial_run_frequency"]
print(f"Partial-run frequency: {prf['rate']:.1%} ({prf['partial_runs']}/{prf['total_runs']} runs)")
print(f"Schema failure rate:   {report['schema_failure_rate']['rate']} "
      f"({report['schema_failure_rate']['total_calls']} calls logged)")
print(f"API call failure rate: {report['api_call_failure_rate']['rate']} "
      f"({report['api_call_failure_rate']['total_calls']} calls logged)")
print(f"Important missed-risk rate: {report['important_missed_risk_rate']['rate']} "
      f"— {report['important_missed_risk_rate']['note']}")

print("\nCost/latency by stage:")
for stage, stats in report["cost_and_latency_by_stage"]["stages"].items():
    print(f"  {stage:<24} {stats['calls']:>4} calls  "
          f"avg_latency={stats['avg_latency_ms']:>7.1f}ms  "
          f"total_tokens={stats['total_tokens']:>8,}  "
          f"avg_tokens/call={stats['avg_tokens_per_call']:>7.1f}")

Evaluation report saved to: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\artifacts\reports\phase7_evaluation_report.json
Parse success:        100.0% (36/36)
Entity accuracy:       100.0% (20/20 human-reviewed)
Retrieval relevance:   100.0% (recall@5, 14 questions)
Citation validity:     94.6% (122/129 findings verified, 7 failed, across 15 documents)
Partial-run frequency: 0.0% (0/45 runs)
Schema failure rate:   0.0 (681 calls logged)
API call failure rate: 0.0 (681 calls logged)
Important missed-risk rate: None — Not independently measurable without a hand-reviewed expected-risk checklist per document (not yet built for Legal/Financial/Operational the way Phase 3 built one for retrieval). One concretely documented gap exists: Phase 3's confidentiality-definition retrieval miss would propagate to the Legal agent for any topic relying on that chunk.

Cost/latency by stage:
 

## 3. Generate versioned, evidence-linked document reports

One Markdown report per document, built from the already-saved Phase 5
`Assessment` — findings sorted by severity, every finding's evidence and
uncertainty flag, unresolved issues, consensus recommendations, and
document/model/prompt/policy version + run ID. Nothing is re-derived or
invented here; this is a presentation layer over already-verified data.

In [10]:
report_paths = generate_all_reports()
print("\nSample report (MLA-001), first 25 lines:\n")
print("\n".join(generate_document_report("MLA-001").splitlines()[:25]))

Generated 15 report(s).

Sample report (MLA-001), first 25 lines:

# ContractIQ Risk Report — MLA-001

**Overall risk:** HIGH  
**Assessment ID:** assessment-MLA-001-29c9c0f9  
**Run ID:** f047cc3ae110  
**Document version:** 1  
**Model version:** gpt-4o-mini  
**Prompt version:** phase5-specialist-prompts-v1  
**Policy version:** risk-policy-v1  
**Created at:** 2026-10-06T01:56:44.582515  
**Report generated:** 2026-10-05T20:32:45.001178+00:00

## Specialist statuses
- **operational**: OK, 5 finding(s), retry_count=0
- **financial**: OK, 4 finding(s), retry_count=0
- **legal**: OK, 7 finding(s), retry_count=0

## Findings (by severity)
- **[HIGH] Scope of Work Defined** (confidence=0.95)
  - rationale: The evidence clearly outlines the scope of services to be provided under the agreement, detailing the responsibilities and deliverables of the Vendor.
  - evidence: `['MLA-001-c0']` — "The scope of services covered under this Agreement encompasses the comprehensive supply, delivery, i

## 4. Langfuse scoring (roadmap item 5)

`agents.py`'s `_verify_and_build_finding` already calls
`langfuse_context.score_current_observation(name="citation_validity", ...)`
for every finding, scored by the independently-computed verification
outcome — not the model's own confidence, per the roadmap's explicit
instruction. This is wired at the source (the real observation that
produced the finding), not backfilled here. Confirm the hook exists and
review what it does:


In [11]:
import inspect
from src.contractiq.agents import _verify_and_build_finding

source = inspect.getsource(_verify_and_build_finding)
scoring_block = source[source.index("if langfuse_context"):source.index("return Finding")]
print(scoring_block)


if langfuse_context is not None:
        try:
            langfuse_context.score_current_observation(
                name="citation_validity",
                value=1.0 if status == VerificationStatus.VERIFIED else 0.0,
                comment=f"verification_status={status.value}",
            )
        except Exception:  # noqa: BLE001 - scoring must never break finding creation
            pass

    


## 5. Focused unit tests (roadmap item 7)

Four real pytest files in `tests/`, covering exactly the roadmap's list —
source mapping, entity normalization, index idempotency, consensus
behavior. `test_source_mapping.py` and `test_index_idempotency.py` run
against real saved artifacts / a fake in-memory collection with **no
OpenAI calls needed**; `test_entity_normalization.py` and
`test_consensus.py` are regression tests for the two real bugs this phase
found and fixed.


In [12]:
import subprocess

# Use an absolute path to the .contract venv's python, not a relative one —
# this notebook's kernel can be a different interpreter (e.g. Anaconda) than
# the one running tests/contractiq, so relative-path resolution is fragile.
venv_python = config.PROJECT_ROOT.parent / ".contract" / "Scripts" / "python.exe"
assert venv_python.exists(), f"venv python not found at {venv_python}"

result = subprocess.run(
    [str(venv_python), "-m", "pytest", "tests/", "-v"],
    cwd=str(config.PROJECT_ROOT), capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-2000:])


============================= test session starts =============================
platform win32 -- Python 3.12.7, pytest-9.1.1, pluggy-1.6.0 -- E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\.contract\Scripts\python.exe
cachedir: .pytest_cache
rootdir: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT
plugins: anyio-4.15.1
collecting ... collected 11 items

tests/test_consensus.py::test_critical_finding_is_never_averaged_away PASSED [  9%]
tests/test_consensus.py::test_failed_dimension_appears_in_unresolved_issues PASSED [ 18%]
tests/test_consensus.py::test_only_genuinely_equivalent_findings_are_deduplicated PASSED [ 27%]
tests/test_entity_normalization.py::test_short_keyword_does_not_match_inside_unrelated_words PASSED [ 36%]
tests/test_entity_normalization.py::test_short_keyword_still_matches

## Honest status of Phase 7 at this point

- **Built:** `evaluation.py` (8 metrics aggregated from real saved
  artifacts, `build_evaluation_report()`), `reporting.py` (versioned,
  evidence-linked per-document reports from Phase 5 Assessments), a real
  `instrumentation` log (`phase7_llm_calls.jsonl`, written by
  `agents.py`/`retrieval.py` as calls actually happen — latency, token
  usage, schema failures, API errors), a Langfuse `citation_validity`
  score wired into the real observation that produces each finding, and
  4 pytest files covering source mapping, entity normalization, index
  idempotency, and consensus behavior.
- **Two real bugs found and fixed while building this phase, not assumed
  working:**
  1. `parse_success_rate` read a stale Phase 0 manifest field never
     updated after parsing — silently reported 0% instead of the real
     100%. Fixed to check the actual `parsed_documents/` output.
  2. Phase 2's money/date context classification used the whole clause
     as its window, letting the word "Invoice" on one line relabel a GST
     amount on an adjacent line as `invoice_amount`. Fixed with a
     line-aware local window; verified against the real FIN-003 document
     before and after.
- **Known, reported limitation — not fabricated:** `important_missed_risk_rate`
  has no ground truth to measure against yet (would need a hand-reviewed
  expected-risk checklist per document, the same kind of manual work Phase 3
  did for retrieval) — reports `null` with an explicit note rather than a
  guessed number.
- **Operational blocker resolved:** OpenAI credits were topped up and a full
  Phase 5 batch (15 docs × 3 specialists) was re-run. Real numbers now
  populate every metric that was previously `null`: 681 LLM calls logged,
  0 schema failures, 0 API failures, citation validity 94.6%
  (122/129 findings verified), and real per-stage latency/token cost
  (embedding ~349ms avg; specialists ~1.3–1.5s avg, ~2,000 tokens/call).
- **Not built yet (Phase 8):** the Gradio UI.

